In [2]:
import math

class TicTacToe:
    def __init__(self):
        self.board = [' '] * 9
        self.nodes_generated = 0
        self.cutoffs = 0
        self.stage_stats = []

    def print_board(self):
        for i in range(0, 9, 3):
            print(f" {self.board[i]} | {self.board[i+1]} | {self.board[i+2]} ")
            if i < 6: print("---+---+---")
        print()

    def check_winner(self):
        win_states = [(0,1,2), (3,4,5), (6,7,8), (0,3,6), (1,4,7), (2,5,8), (0,4,8), (2,4,6)]
        for a, b, c in win_states:
            if self.board[a] == self.board[b] == self.board[c] != ' ':
                return self.board[a]
        if ' ' not in self.board:
            return 'Draw'
        return None

    def minimax(self, depth, is_maximizing, alpha, beta):
        self.nodes_generated += 1
        winner = self.check_winner()

        if winner == 'X': return 10 - depth
        if winner == 'O': return depth - 10
        if winner == 'Draw': return 0

        if is_maximizing:
            max_eval = -math.inf
            for i in range(9):
                if self.board[i] == ' ':
                    self.board[i] = 'X'
                    eval = self.minimax(depth + 1, False, alpha, beta)
                    self.board[i] = ' '
                    max_eval = max(max_eval, eval)
                    alpha = max(alpha, eval)
                    if beta <= alpha:
                        self.cutoffs += 1
                        break
            return max_eval
        else:
            min_eval = math.inf
            for i in range(9):
                if self.board[i] == ' ':
                    self.board[i] = 'O'
                    eval = self.minimax(depth + 1, True, alpha, beta)
                    self.board[i] = ' '
                    min_eval = min(min_eval, eval)
                    beta = min(beta, eval)
                    if beta <= alpha:
                        self.cutoffs += 1
                        break
            return min_eval

    def best_move(self, player):
        self.nodes_generated = 0
        self.cutoffs = 0
        best_val = -math.inf if player == 'X' else math.inf
        move = -1

        for i in range(9):
            if self.board[i] == ' ':
                self.board[i] = player
                move_val = self.minimax(0, player == 'O', -math.inf, math.inf)
                self.board[i] = ' '

                if player == 'X' and move_val > best_val:
                    best_val, move = move_val, i
                elif player == 'O' and move_val < best_val:
                    best_val, move = move_val, i
        return move

    def play_game(self):
        current_player = 'X'
        stage = 1
        self.print_board()

        while not self.check_winner():
            print(f"--- Turn {stage} ---")

            if current_player == 'X':
                valid_move = False
                while not valid_move:
                    try:
                        move = int(input("Enter your move (0-8): "))
                        if 0 <= move <= 8 and self.board[move] == ' ':
                            valid_move = True
                        else:
                            print("Invalid move.")
                    except ValueError:
                        print("Invalid input.")
            else:
                move = self.best_move(current_player)
                self.stage_stats.append({
                    'Stage': stage,
                    'Player': 'AI (O)',
                    'Nodes': self.nodes_generated,
                    'Cutoffs': self.cutoffs
                })

            self.board[move] = current_player
            self.print_board()
            current_player = 'O' if current_player == 'X' else 'X'
            stage += 1

        winner = self.check_winner()
        if winner == 'Draw':
            print("Game Over. Draw!")
        else:
            print(f"Game Over. Winner: {winner}")

        self.display_statistics()

    def display_statistics(self):
        print("\n" + "="*45)
        print(f"{'Stage':<8} | {'Player':<8} | {'Nodes Gen':<10} | {'Cutoffs':<8}")
        print("-" * 45)
        for stat in self.stage_stats:
            print(f"{stat['Stage']:<8} | {stat['Player']:<8} | {stat['Nodes']:<10} | {stat['Cutoffs']:<8}")
        print("="*45)

print("Tic-Tac-Toe:(X) vs (O)")
game = TicTacToe()
game.play_game()

Tic-Tac-Toe:(X) vs (O)
   |   |   
---+---+---
   |   |   
---+---+---
   |   |   

--- Turn 1 ---
Enter your move (0-8): 1
   | X |   
---+---+---
   |   |   
---+---+---
   |   |   

--- Turn 2 ---
 O | X |   
---+---+---
   |   |   
---+---+---
   |   |   

--- Turn 3 ---
Enter your move (0-8): 3
 O | X |   
---+---+---
 X |   |   
---+---+---
   |   |   

--- Turn 4 ---
 O | X |   
---+---+---
 X | O |   
---+---+---
   |   |   

--- Turn 5 ---
Enter your move (0-8): 8
 O | X |   
---+---+---
 X | O |   
---+---+---
   |   | X 

--- Turn 6 ---
 O | X | O 
---+---+---
 X | O |   
---+---+---
   |   | X 

--- Turn 7 ---
Enter your move (0-8): 6
 O | X | O 
---+---+---
 X | O |   
---+---+---
 X |   | X 

--- Turn 8 ---
 O | X | O 
---+---+---
 X | O |   
---+---+---
 X | O | X 

--- Turn 9 ---
Enter your move (0-8): 5
 O | X | O 
---+---+---
 X | O | X 
---+---+---
 X | O | X 

Game Over. Draw!

Stage    | Player   | Nodes Gen  | Cutoffs 
---------------------------------------------

In [3]:
import heapq

class PuzzleState:
    def __init__(self, board, goal, moves=0, previous=None):
        self.board = board
        self.goal = goal
        self.moves = moves
        self.previous = previous
        self.blank_pos = self.board.index(0)

    def h1_misplaced(self):
        return sum(1 for i in range(9) if self.board[i] != 0 and self.board[i] != self.goal[i])

    def h2_manhattan(self):
        distance = 0
        for i in range(9):
            if self.board[i] != 0:
                x1, y1 = divmod(i, 3)
                x2, y2 = divmod(self.goal.index(self.board[i]), 3)
                distance += abs(x1 - x2) + abs(y1 - y2)
        return distance

    def get_neighbors(self):
        neighbors = []
        x, y = divmod(self.blank_pos, 3)
        directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < 3 and 0 <= ny < 3:
                new_blank = nx * 3 + ny
                new_board = list(self.board)
                new_board[self.blank_pos], new_board[new_blank] = new_board[new_blank], new_board[self.blank_pos]
                neighbors.append(PuzzleState(tuple(new_board), self.goal, self.moves + 1, self))
        return neighbors

    def __lt__(self, other):
        return False

def solve_8_puzzle(start_board, goal_board, heuristic_type):
    start_state = PuzzleState(start_board, goal_board)
    frontier = []
    state_id = 0

    if heuristic_type == "Manhattan":
        h_val = start_state.h2_manhattan()
    else:
        h_val = start_state.h1_misplaced()

    heapq.heappush(frontier, (h_val, state_id, start_state))
    explored = set()
    nodes_generated = 0

    while frontier:
        _, _, current = heapq.heappop(frontier)
        if current.board == goal_board:
            return current, nodes_generated
        explored.add(current.board)
        for neighbor in current.get_neighbors():
            nodes_generated += 1
            if neighbor.board not in explored:
                if heuristic_type == "Manhattan":
                    h = neighbor.h2_manhattan()
                else:
                    h = neighbor.h1_misplaced()
                f = neighbor.moves + h
                state_id += 1
                heapq.heappush(frontier, (f, state_id, neighbor))
                explored.add(neighbor.board)
    return None, nodes_generated

def get_user_board(prompt):
    print(prompt)
    while True:
        try:
            user_input = input("> ")
            board = tuple(map(int, user_input.split()))
            if len(board) == 9 and set(board) == set(range(9)):
                return board
            print("Invalid input. Enter 9 unique numbers from 0 to 8.")
        except ValueError:
            print("Invalid input. Enter numbers only.")

print("--- 8-PUZZLE HEURISTIC COMPARISON ---")
start = get_user_board("Define START state (9 numbers separated by space):")
goal = get_user_board("Define GOAL state (9 numbers separated by space):")

heuristics = ["Manhattan", "Misplaced Tiles"]
results = []

for h in heuristics:
    print(f"\nRunning A* with {h} heuristic...")
    solution, nodes = solve_8_puzzle(start, goal, h)
    moves = solution.moves if solution else "N/A"
    results.append({
        'Heuristic': h,
        'Moves': moves,
        'Nodes Generated': nodes
    })

print("\n" + "="*55)
print(f"{'Heuristic':<20} | {'Moves':<8} | {'Nodes Generated':<15}")
print("-" * 55)
for res in results:
    print(f"{res['Heuristic']:<20} | {str(res['Moves']):<8} | {str(res['Nodes Generated']):<15}")
print("="*55)

--- 8-PUZZLE HEURISTIC COMPARISON ---
Define START state (9 numbers separated by space):
> 1
Invalid input. Enter 9 unique numbers from 0 to 8.
> 1 2 3 4 5 6 7 8 9 0
Invalid input. Enter 9 unique numbers from 0 to 8.
> 0 1 2 3 4 5 6 7 8 9 
Invalid input. Enter 9 unique numbers from 0 to 8.
> 1 2 3 4 5 6 7 8 0
Define GOAL state (9 numbers separated by space):
> 1 2 3 4 5 6 7 8 0

Running A* with Manhattan heuristic...

Running A* with Misplaced Tiles heuristic...

Heuristic            | Moves    | Nodes Generated
-------------------------------------------------------
Manhattan            | 0        | 0              
Misplaced Tiles      | 0        | 0              


In [6]:
import heapq
import math


# ============================================================
#                    PART 1: 8-PUZZLE
# ============================================================

class PuzzleState:

    def __init__(self, board, goal, moves=0, parent=None):
        self.board = board
        self.goal = goal
        self.moves = moves
        self.parent = parent
        self.blank = board.index(0)

    # Heuristic 1: Misplaced Tiles
    def misplaced_tiles(self):

        return sum(
            1
            for i in range(9)
            if self.board[i] != 0
            and self.board[i] != self.goal[i]
        )

    # Heuristic 2: Manhattan Distance
    def manhattan_distance(self):

        distance = 0

        for i in range(9):

            tile = self.board[i]

            if tile != 0:

                row, col = divmod(i, 3)

                goal_index = self.goal.index(tile)

                goal_row, goal_col = divmod(
                    goal_index, 3
                )

                distance += abs(row - goal_row)
                distance += abs(col - goal_col)

        return distance

    # Generate neighboring states
    def get_neighbors(self):

        neighbors = []

        row, col = divmod(self.blank, 3)

        directions = [
            (-1, 0),     # Up
            (1, 0),      # Down
            (0, -1),     # Left
            (0, 1)       # Right
        ]

        for dr, dc in directions:

            new_row = row + dr
            new_col = col + dc

            if 0 <= new_row < 3 and 0 <= new_col < 3:

                new_blank = new_row * 3 + new_col

                new_board = list(self.board)

                new_board[self.blank], new_board[new_blank] = \
                    new_board[new_blank], new_board[self.blank]

                neighbors.append(
                    PuzzleState(
                        tuple(new_board),
                        self.goal,
                        self.moves + 1,
                        self
                    )
                )

        return neighbors


def a_star(start, goal, heuristic):

    start_state = PuzzleState(start, goal)

    if heuristic == "Misplaced Tiles":
        h = start_state.misplaced_tiles()
    else:
        h = start_state.manhattan_distance()

    frontier = []

    counter = 0

    heapq.heappush(
        frontier,
        (h, counter, start_state)
    )

    explored = set()

    nodes_generated = 0

    while frontier:

        _, _, current = heapq.heappop(frontier)

        if current.board == goal:
            return current, nodes_generated

        if current.board in explored:
            continue

        explored.add(current.board)

        for neighbor in current.get_neighbors():

            nodes_generated += 1

            if neighbor.board in explored:
                continue

            if heuristic == "Misplaced Tiles":

                h = neighbor.misplaced_tiles()

            else:

                h = neighbor.manhattan_distance()

            # A* formula
            f = neighbor.moves + h

            counter += 1

            heapq.heappush(
                frontier,
                (f, counter, neighbor)
            )

    return None, nodes_generated


def get_puzzle_board(message):

    while True:

        try:

            board = tuple(
                map(int, input(message).split())
            )

            if (
                len(board) == 9
                and set(board) == set(range(9))
            ):
                return board

            print(
                "Enter 9 unique numbers from 0 to 8."
            )

        except ValueError:

            print("Enter numbers only.")


def print_puzzle_solution(solution):

    path = []

    current = solution

    while current:

        path.append(current.board)
        current = current.parent

    path.reverse()

    print("\nSolution Path:")

    for step, board in enumerate(path):

        print("\nStep", step)

        for i in range(0, 9, 3):

            print(
                board[i],
                board[i + 1],
                board[i + 2]
            )


def run_8_puzzle():

    print("\n")
    print("=" * 60)
    print("              8-PUZZLE USING A*")
    print("=" * 60)

    start = get_puzzle_board(
        "Enter START state (0 = blank): "
    )

    goal = get_puzzle_board(
        "Enter GOAL state (0 = blank): "
    )

    heuristics = [
        "Misplaced Tiles",
        "Manhattan Distance"
    ]

    results = []

    for heuristic in heuristics:

        print(
            "\nRunning A* using",
            heuristic
        )

        solution, nodes = a_star(
            start,
            goal,
            heuristic
        )

        if solution:

            print("Solution found!")
            print("Moves:", solution.moves)
            print("Nodes Generated:", nodes)

            results.append([
                heuristic,
                solution.moves,
                nodes
            ])

        else:

            print("No solution found.")

            results.append([
                heuristic,
                "N/A",
                nodes
            ])

    print("\n")
    print("=" * 60)
    print("              HEURISTIC COMPARISON")
    print("=" * 60)

    print(
        f"{'Heuristic':<25}"
        f"{'Moves':<10}"
        f"{'Nodes Generated':<20}"
    )

    print("-" * 60)

    for result in results:

        print(
            f"{result[0]:<25}"
            f"{str(result[1]):<10}"
            f"{result[2]:<20}"
        )

    print("=" * 60)


# ============================================================
#                 PART 2: TIC-TAC-TOE
# ============================================================

board = [" "] * 9

nodes_generated = 0
cutoffs = 0

search_table = []


def display_board():

    print()

    for i in range(9):

        value = board[i]

        if value == " ":
            value = str(i + 1)

        print(f" {value} ", end="")

        if i % 3 != 2:
            print("|", end="")

        if i % 3 == 2 and i != 8:
            print("\n---+---+---")

    print()


def check_winner():

    winning_positions = [

        (0, 1, 2),
        (3, 4, 5),
        (6, 7, 8),

        (0, 3, 6),
        (1, 4, 7),
        (2, 5, 8),

        (0, 4, 8),
        (2, 4, 6)
    ]

    for a, b, c in winning_positions:

        if (
            board[a] != " "
            and board[a] == board[b]
            and board[b] == board[c]
        ):

            return board[a]

    if " " not in board:
        return "Draw"

    return None


def minimax(depth, maximizing, alpha, beta):

    global nodes_generated
    global cutoffs
    global search_table

    nodes_generated += 1

    result = check_winner()

    # Terminal state
    if result == "O":

        return 10 - depth

    if result == "X":

        return depth - 10

    if result == "Draw":

        return 0

    # --------------------------------------------------------
    # MAX PLAYER - O
    # --------------------------------------------------------

    if maximizing:

        best_value = -math.inf

        for i in range(9):

            if board[i] == " ":

                board[i] = "O"

                value = minimax(
                    depth + 1,
                    False,
                    alpha,
                    beta
                )

                board[i] = " "

                best_value = max(
                    best_value,
                    value
                )

                alpha = max(
                    alpha,
                    best_value
                )

                cutoff = "YES" if alpha >= beta else "NO"

                search_table.append([
                    depth,
                    "MAX",
                    i + 1,
                    value,
                    alpha,
                    beta,
                    cutoff
                ])

                if alpha >= beta:

                    cutoffs += 1

                    break

        return best_value

    # --------------------------------------------------------
    # MIN PLAYER - X
    # --------------------------------------------------------

    else:

        best_value = math.inf

        for i in range(9):

            if board[i] == " ":

                board[i] = "X"

                value = minimax(
                    depth + 1,
                    True,
                    alpha,
                    beta
                )

                board[i] = " "

                best_value = min(
                    best_value,
                    value
                )

                beta = min(
                    beta,
                    best_value
                )

                cutoff = "YES" if alpha >= beta else "NO"

                search_table.append([
                    depth,
                    "MIN",
                    i + 1,
                    value,
                    alpha,
                    beta,
                    cutoff
                ])

                if alpha >= beta:

                    cutoffs += 1

                    break

        return best_value


def find_best_move():

    global nodes_generated
    global cutoffs
    global search_table

    nodes_generated = 0
    cutoffs = 0
    search_table = []

    best_value = -math.inf
    best_move = None

    alpha = -math.inf
    beta = math.inf

    for i in range(9):

        if board[i] == " ":

            board[i] = "O"

            value = minimax(
                0,
                False,
                alpha,
                beta
            )

            board[i] = " "

            if value > best_value:

                best_value = value
                best_move = i

            alpha = max(
                alpha,
                best_value
            )

    return best_move


def print_search_table():

    print("\n")
    print("=" * 95)
    print("              MINIMAX + ALPHA-BETA TABLE")
    print("=" * 95)

    print(
        f"{'Depth':<8}"
        f"{'Stage':<10}"
        f"{'Move':<8}"
        f"{'Value':<10}"
        f"{'Alpha':<12}"
        f"{'Beta':<12}"
        f"{'Cutoff':<10}"
    )

    print("-" * 95)

    for row in search_table:

        depth, stage, move, value, alpha, beta, cutoff = row

        print(
            f"{depth:<8}"
            f"{stage:<10}"
            f"{move:<8}"
            f"{value:<10}"
            f"{str(alpha):<12}"
            f"{str(beta):<12}"
            f"{cutoff:<10}"
        )

    print("=" * 95)

    print("Total Nodes Generated :", nodes_generated)
    print("Total Cutoffs         :", cutoffs)


def play_tic_tac_toe():

    global board

    board = [" "] * 9

    print("\n")
    print("=" * 60)
    print("              TIC-TAC-TOE")
    print("=" * 60)

    print("You are X")
    print("Computer is O")

    while True:

        display_board()

        # ----------------------------------------------------
        # HUMAN MOVE
        # ----------------------------------------------------

        while True:

            try:

                move = int(
                    input("Enter your move (1-9): ")
                ) - 1

                if (
                    0 <= move < 9
                    and board[move] == " "
                ):

                    board[move] = "X"
                    break

                print("Invalid move.")

            except ValueError:

                print(
                    "Enter a number from 1 to 9."
                )

        result = check_winner()

        if result is not None:

            display_board()

            if result == "X":

                print("You win!")

            else:

                print("Game Draw!")

            return

        # ----------------------------------------------------
        # COMPUTER MOVE
        # ----------------------------------------------------

        print("\nComputer is thinking...")

        best_move = find_best_move()

        board[best_move] = "O"

        print(
            "Computer selected position:",
            best_move + 1
        )

        result = check_winner()

        if result is not None:

            display_board()

            if result == "O":

                print("Computer wins!")

            else:

                print("Game Draw!")

            print_search_table()

            return


# ============================================================
#                     MAIN PROGRAM
# ============================================================


# First execute 8-Puzzle
run_8_puzzle()

# Then automatically start Tic-Tac-Toe
play_tic_tac_toe()



              8-PUZZLE USING A*
Enter START state (0 = blank): 1 2 3 4 5 6 7 8 0
Enter GOAL state (0 = blank): 1 2 3 4 5 6 7 8 0

Running A* using Misplaced Tiles
Solution found!
Moves: 0
Nodes Generated: 0

Running A* using Manhattan Distance
Solution found!
Moves: 0
Nodes Generated: 0


              HEURISTIC COMPARISON
Heuristic                Moves     Nodes Generated     
------------------------------------------------------------
Misplaced Tiles          0         0                   
Manhattan Distance       0         0                   


              TIC-TAC-TOE
You are X
Computer is O

 1 | 2 | 3 
---+---+---
 4 | 5 | 6 
---+---+---
 7 | 8 | 9 
Enter your move (1-9): 1

Computer is thinking...
Computer selected position: 5

 X | 2 | 3 
---+---+---
 4 | O | 6 
---+---+---
 7 | 8 | 9 
Enter your move (1-9): 2

Computer is thinking...
Computer selected position: 3

 X | X | O 
---+---+---
 4 | O | 6 
---+---+---
 7 | 8 | 9 
Enter your move (1-9): 3
Invalid move.
Enter your